# Verify both nozzle block encodings

All ten original code stages, the original 8 by 8 nozzle matrix with 21 entries, both Pauli and banded block encodings, fresh signed right-hand sides, and both unnormalized projections checked against the scaled matrix product.

The pinned CSR matrix is stored as plain numeric JSON. Pauli and diagonal decompositions are recomputed locally. Native synthesis prepares the data and selector amplitudes; the public Circuit composition applies Pauli SELECT or a bounded signed banded dilation, then unprepares the selector. These encodings have their own reported LCU scale and resources; no historical Gray-code gate count is claimed. Fixed local seeds make the generated right-hand sides reproducible. Absolute amplitudes are compared directly, with no sign correction or postselection renormalization.

## Verify an unnormalized block action, not only its state direction

The visible banded factory constructs explicit FlagQuantum Circuit kernels;
the projection wrapper chooses Pauli or banded encoding and executes it.
The host supplies the sparse matrix, band offsets and normalization. This
workflow does not yet ask native qfunc to infer an encoding from any matrix.
The direct Circuit code remains visible rather than being labeled automatic
high-level synthesis.

Try the two existing nonzero right-hand-side seeds and compare the full
selector-zero amplitudes with A @ rhs divided by scale. Do not normalize both
vectors first: that would hide a wrong block scale. Pauli and banded widths
and operation counts describe different encodings of the same declared action,
not proof of hardware speed or quantum advantage.

## Logical Pauli block composition

The additional qfunc below expresses PREPARE / SELECT / PREPARE-adjoint using named selector and data ports. The classical model produces the same 24 Pauli terms, their magnitudes and complex coefficient phases. Synthesis lowers bounded selector predicates and controlled borrowed helpers, and checks the inverse preparation. It does not replace the source matrix or compute the projected answer classically.

The native program uses eight output bits plus four clean work bits (12 total), compared with the original eight-bit Circuit SELECT. Both actual costs are reported; the tighter eight-bit native budget is explicitly refused. The banded route remains the original Circuit implementation. Projected amplitudes are unnormalized, so their squared norm is the success probability, not a normalized solution distribution.


## Use the local FlagQuantum workflow

In [ ]:
import flagquantum as fq
import torch

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.quantum import (
    CTYPE,
    bands,
    preparation,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.workflow import (
    CTYPE,
    Encoding,
    Projection,
    pauli_encoding,
)


def banded_encoding(
    matrix: torch.Tensor, rhs: torch.Tensor
) -> tuple[fq.Circuit, float]:
    offsets = bands(matrix)
    if offsets != (-1, 0, 1) or bool(matrix.imag.any()):
        raise ValueError(
            "this banded encoding requires the real source tridiagonal profile"
        )
    widths = [float(matrix.diagonal(offset).abs().max()) for offset in offsets]
    alpha = sum(widths)
    weights = preparation(
        torch.tensor([(w / alpha) ** 0.5 for w in widths] + [0], dtype=torch.float64)
    )
    matrix_parts = []
    for offset, scale in zip(offsets, widths, strict=True):
        diagonal = torch.zeros(8, dtype=torch.float64)
        shift = torch.zeros((8, 8), dtype=CTYPE)
        for col in range(8):
            row = col - offset
            if 0 <= row < 8:
                diagonal[col] = matrix[row, col].real / scale
            shift[row % 8, col] = 1
        d = torch.diag(diagonal).to(CTYPE)
        complement = torch.diag((1 - diagonal.square()).clamp_min(0).sqrt()).to(CTYPE)
        dilated = torch.cat(
            (torch.cat((d, complement), 1), torch.cat((complement, -d), 1)), 0
        )
        both = torch.zeros((16, 16), dtype=CTYPE)
        both[:8, :8] = both[8:, 8:] = shift
        matrix_parts.append(both @ dilated)
    matrix_parts.append(torch.eye(16, dtype=CTYPE))
    select = torch.zeros((64, 64), dtype=CTYPE)
    for index, part in enumerate(matrix_parts):
        group = slice(16 * index, 16 * (index + 1))
        select[group, group] = part
    torch.testing.assert_close(
        select.mH @ select, torch.eye(64, dtype=CTYPE), atol=1e-12, rtol=0
    )
    circuit = (
        fq.Circuit(6, dtype=CTYPE)
        .compose(weights, qubits=(0, 1))
        .compose(preparation(rhs), qubits=(3, 4, 5))
    )
    circuit.any(*range(6), matrix=select)
    circuit.compose(weights.adjoint(), qubits=(0, 1))
    return circuit, alpha


def project_preview(
    method: Encoding, matrix: torch.Tensor, vector: torch.Tensor
) -> Projection:
    if method not in ("pauli", "banded"):
        raise ValueError("choose pauli or banded")
    encoder = pauli_encoding if method == "pauli" else banded_encoding
    circuit, scale = encoder(matrix, vector)
    state = fq.run(circuit).statevector().flatten()
    values = state[:8]
    expected = matrix @ vector.to(CTYPE) / scale
    torch.testing.assert_close(values, expected, rtol=0, atol=1e-10)
    ir = circuit.to_ir()
    return Projection(
        method,
        ir.n_wires,
        len(ir.instructions),
        scale,
        values,
        expected,
        float((values - expected).abs().max()),
    )


from pathlib import Path

from IPython.display import Image, display

import cmath
import math

from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    QNum,
    X,
    allocate,
    apply,
    control,
    inplace_prepare_amplitudes,
    phase,
    prepare_amplitudes,
    qfunc,
    select_rotation,
    within,
)
import examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.quantum as native_quantum


@qfunc
def pauli_word(codes: list[float], term: int, data: QArray[QBit]) -> None:
    for bit in range(3):
        if codes[3 * term + bit] == 1:
            X(data[bit])
        if codes[3 * term + bit] == 2:
            phase(data[bit], -math.pi / 2)
            X(data[bit])
            phase(data[bit], math.pi / 2)
        if codes[3 * term + bit] == 3:
            phase(data[bit], math.pi)


@qfunc
def pauli_block(
    amplitudes: list[float],
    weights: list[float],
    codes: list[float],
    angles: list[float],
    count: int,
    selection: Output[QNum[5]],
    data: Output[QArray[QBit]],
) -> None:
    prepare_amplitudes(amplitudes, 0, data)
    allocate(selection)
    with within():
        inplace_prepare_amplitudes(weights, 0, selection)
    with apply():
        for term in range(count):
            if codes[3 * term] + codes[3 * term + 1] + codes[3 * term + 2] > 0:
                control(selection == term, lambda: pauli_word(codes, term, data))
            control(selection == term, lambda: phase(1, angles[term]))


native_quantum.pauli_block = pauli_block


## Import preparation and verification tools

In [ ]:
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.model import (
    bands,
    pauli_expansion,
    right_hand_side,
    source_matrix,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.plots import (
    draw,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.verify_block_encoding.overview.workflow import (
    project,
)


## Define the checked projection

In [ ]:
print("Apply the block encoding, then read only the selector-zero amplitudes.")


## Define the independent reference

In [ ]:
print("Compare the unnormalized projected amplitudes directly with A @ rhs / scale.")


## Choose the output directory

In [ ]:
folder = Path("artifacts/development/cfd-block-encoding")


## Load the original nozzle matrix

In [ ]:
matrix = source_matrix()
print(matrix.shape, matrix.count_nonzero().item())


## Regenerate and execute Pauli SELECT

In [ ]:
terms = pauli_expansion(matrix)
print("Pauli terms:", len(terms))
pauli_result = project_preview("pauli", matrix, right_hand_side(37))

# Execute the visible logical program without changing the original Circuit path.
native, native_scale = native_quantum.native_pauli_program(matrix, right_hand_side(37))
native_result = fq.run(native)
native_state = native_result.statevector().reshape(256, 16)
circuit_reference, _ = pauli_encoding(matrix, right_hand_side(37))
torch.testing.assert_close(
    native_state[:, 0],
    fq.run(circuit_reference).statevector().flatten(),
    atol=1e-12,
    rtol=0,
)
assert torch.count_nonzero(native_state[:, 1:]) == 0
torch.testing.assert_close(
    native_state[:8, 0],
    matrix @ right_hand_side(37).to(CTYPE) / native_scale,
    atol=1e-12,
    rtol=0,
)
print("Native resources (includes four clean work bits):", native.report["resources"])
print(
    "Circuit SELECT resources:",
    circuit_reference.to_ir().n_wires,
    len(circuit_reference.to_ir().instructions),
)
print(
    "Named selector-zero success probability:",
    sum(
        probability
        for (selector, _), probability in native.probabilities(
            native_result, registers=("selection", "data")
        ).items()
        if selector == 0
    ),
)


## Compare the Pauli projection

In [ ]:
print("Pauli scale and error:", pauli_result.scale, pauli_result.max_error)
assert pauli_result.max_error < 1e-10
display(Image(filename=str(draw(pauli_result, folder / "pauli.png"))))


## Regenerate and execute the banded encoding

The native program below dilates each signed diagonal value with a conditional
rotation, then shifts the data register modulo eight. PREPARE and its inverse
combine the three bands with the original LCU scale. The dense Circuit above
is an independent reference; the native program does not embed its matrix.
Eight native qubits include two work bits, compared with six for Circuit.
These are different instruction vocabularies, not a basis-level efficiency comparison.
A seven-qubit constraint selects one work bit and more logical instructions.


In [ ]:
print("Offsets:", bands(matrix))
banded_result = project_preview("banded", matrix, right_hand_side(91))


@qfunc
def banded_term(angles: list[float], term: int, flag: QBit, data: QNum[3]) -> None:
    # RY(2*acos(d)) Z dilates each real signed diagonal value d.
    phase(flag, math.pi)
    select_rotation(
        "Y",
        [
            angles[8 * term],
            angles[8 * term + 1],
            angles[8 * term + 2],
            angles[8 * term + 3],
            angles[8 * term + 4],
            angles[8 * term + 5],
            angles[8 * term + 6],
            angles[8 * term + 7],
        ],
        data,
        flag,
    )
    if term == 0:
        data += 1
    if term == 2:
        data -= 1


@qfunc
def banded_block(
    amplitudes: list[float],
    weights: list[float],
    angles: list[float],
    selection: Output[QNum[2]],
    flag: Output[QBit],
    data: Output[QNum[3]],
) -> None:
    allocate(selection)
    allocate(flag)
    prepare_amplitudes(amplitudes, 0, data)
    with within():
        inplace_prepare_amplitudes(weights, 0, selection)
    with apply():
        for term in range(3):
            control(
                selection == term,
                lambda: banded_term(angles, term, flag, data),  # noqa: B023
            )

native_quantum.banded_term = banded_term
native_quantum.banded_block = banded_block
native_banded, banded_scale = native_quantum.native_banded_program(
    matrix, right_hand_side(91)
)
native_banded_result = fq.run(native_banded)
native_banded_state = native_banded_result.statevector().reshape(64, 4)
banded_reference, _ = banded_encoding(matrix, right_hand_side(91))
torch.testing.assert_close(
    native_banded_state[:, 0],
    fq.run(banded_reference).statevector().flatten(),
    atol=1e-12,
    rtol=0,
)
assert float(native_banded_state[:, 1:].abs().square().sum()) < 1e-24
torch.testing.assert_close(
    native_banded_state[:8, 0],
    matrix @ right_hand_side(91).to(CTYPE) / banded_scale,
    atol=1e-12,
    rtol=0,
)
print("Native banded resources (two clean work bits):", native_banded.report["resources"])
banded_probabilities = native_banded.probabilities(
    native_banded_result, registers=("selection", "flag", "data")
)
print(
    "Named banded success probability:",
    sum(
        probability
        for (selector, flag, _), probability in banded_probabilities.items()
        if selector == 0 and flag == 0
    ),
)

# Tightening the total-qubit budget selects a different verified implementation.
compact_banded, _ = native_quantum.native_banded_program(
    matrix, right_hand_side(91), max_qubits=7
)
compact_result = fq.run(compact_banded)
compact_state = compact_result.statevector().reshape(64, 2)
torch.testing.assert_close(compact_state[:, 0], native_banded_state[:, 0], atol=1e-12, rtol=0)
assert float(compact_state[:, 1:].abs().square().sum()) < 1e-24
print("Tighter qubit budget resources:", compact_banded.report["resources"])


## Compare the banded projection

In [ ]:
print("Banded scale and error:", banded_result.scale, banded_result.max_error)
assert banded_result.max_error < 1e-10
display(Image(filename=str(draw(banded_result, folder / "banded.png"))))
